In [ ]:
from strategies.live_mean_reversion import LiveMeanReversion
from model_filters.catboost_live_model_filter import CatBoostLiveModelFilter
from orders import AlpacaLimitOrderRouter, IBKRLimitOrderRouter
from config import LiveTradingConfig, DEFAULT_LIVE_CONFIG
from execution import build_execution_instrument
from ib_async import IB, Stock, util

# Required for sync ib.connect(...) inside Jupyter/IPython kernels.
util.startLoop()

# ============================================================
# USAGE EXAMPLES
# ============================================================

# These examples assume you already have these files/classes somewhere:
#
# from config.live_trading_config import LiveTradingConfig, DEFAULT_LIVE_CONFIG
# from strategies.live_mean_reversion import LiveMeanReversion
# from routers.ibkr_limit_order_router import IBKRLimitOrderRouter
# from routers.alpaca_limit_order_router import AlpacaLimitOrderRouter
# from model_filters.catboost_live_model_filter import CatBoostLiveModelFilter
#
# And that you already created:
#   ib
#   stock / contract
#   real_time_bars
#
# Example IBKR bars:
#   real_time_bars = ib.reqRealTimeBars(
#       stock,
#       barSize=5,
#       whatToShow="TRADES",
#       useRTH=False,
#   )


# ------------------------------------------------------------
# 1. Default config object
# ------------------------------------------------------------

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 1A. IBKR market-data setup
# ------------------------------------------------------------

IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 42

ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock = Stock(config.symbol, "SMART", "USD")
ib.qualifyContracts(stock)

real_time_bars = ib.reqRealTimeBars(
    stock,
    barSize=5,
    whatToShow="TRADES",
    useRTH=False,
)

execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)


# ------------------------------------------------------------
# 2. DEFAULT: no CatBoost model
#
# Uses AlwaysPassModelFilter automatically.
#
# Model filter automatically returns:
#   ml_enabled = False or default internal value
#   ml_prob = 1.0
#   ml_passed = True
#
# This means the strategy trades only from the BB / median-MAD rules.
# ------------------------------------------------------------

order_router = IBKRLimitOrderRouter(
    ib=ib,
    contract=stock,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    execution_instrument=execution_instrument,
)

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 3. OPTIONAL: CatBoost model filter
#
# This keeps the same BB logic, but adds:
#   model_filter.score(features)
#
# Then should_enter_trade() requires:
#   ml_prob is not None
#   ml_passed == True
# ------------------------------------------------------------

model_cfg = config.model
paths_cfg = config.paths

model_filter = CatBoostLiveModelFilter(
    model_path=paths_cfg["catboost_model_path"],
    feature_cols=model_cfg["feature_cols"],
    prob_threshold=model_cfg.get("prob_threshold", 0.50),
    enabled=model_cfg.get("enabled", True),
)

order_router = IBKRLimitOrderRouter(
    ib=ib,
    contract=stock,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    model_filter=model_filter,
    execution_instrument=execution_instrument,
)

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 4. OPTIONAL: CatBoost disabled from config
#
# Useful if you want to instantiate CatBoostLiveModelFilter,
# but temporarily bypass it without changing strategy code.
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["model"]["enabled"] = False

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)

model_cfg = config.model
paths_cfg = config.paths

model_filter = CatBoostLiveModelFilter(
    model_path=paths_cfg["catboost_model_path"],
    feature_cols=model_cfg["feature_cols"],
    prob_threshold=model_cfg.get("prob_threshold", 0.50),
    enabled=model_cfg.get("enabled", False),
)

order_router = IBKRLimitOrderRouter(
    ib=ib,
    contract=stock,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    model_filter=model_filter,
    execution_instrument=execution_instrument,
)

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 5. OPTIONAL: Alpaca execution with IBKR market data
#
# IBKR provides the live bars.
# Alpaca sends the actual orders.
# ------------------------------------------------------------

order_router = AlpacaLimitOrderRouter(
    client=trading_client,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
)

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 6. OPTIONAL: Alpaca execution + CatBoost model filter
# ------------------------------------------------------------

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)

model_cfg = config.model
paths_cfg = config.paths

model_filter = CatBoostLiveModelFilter(
    model_path=paths_cfg["catboost_model_path"],
    feature_cols=model_cfg["feature_cols"],
    prob_threshold=model_cfg.get("prob_threshold", 0.50),
    enabled=model_cfg.get("enabled", True),
)

order_router = AlpacaLimitOrderRouter(
    client=trading_client,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    model_filter=model_filter,
)

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 7. Quick config edits from notebook
# ------------------------------------------------------------

# Always edit the dict BEFORE creating config:
#
#   config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)
#
# If config already exists, recreate it after editing DEFAULT_LIVE_CONFIG.


# ------------------------------------------------------------
# 7A. Turn CatBoost on
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["model"]["enabled"] = True
DEFAULT_LIVE_CONFIG["model"]["prob_threshold"] = 0.55

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7B. Turn CatBoost off
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["model"]["enabled"] = False

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7C. Use standard Bollinger Bands entry
#
# This is NOT median/MAD based. It uses rolling mean/std:
#   standard_bb_z = (close - rolling_mean) / rolling_std
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["entry_strategy"] = {
    "name": "standard_bb",
    "z": 2.0,
    "double_down_mult": 3.0,
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7D. Use original median/MAD entry
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["entry_strategy"] = {
    "name": "median_mad",
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7E. Custom raw entry conditions
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["entry_conditions"] = {
    "first_entry": [
        {"field": "standard_bb_z", "op": "<=", "value": -2.0},
        {"field": "rsi_14", "op": "<=", "value": 35.0},
    ],
    "double_down": [
        {"field": "standard_bb_z", "op": "<=", "value": -6.0},
    ],
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7F. Use 5-minute signal bars
#
# IB still streams 5-second bars. The strategy aggregates those
# into completed 5-minute signal bars before calculating features.
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["features"]["bar"] = {
    "type": "time",
    "timeframe": "5min",
    "history_window": 78,
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7G. Use 1-minute signal bars
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["features"]["bar"] = {
    "type": "time",
    "timeframe": "1min",
    "history_window": 390,
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7H. Use dollar signal bars
#
# IB still streams 5-second bars. The strategy aggregates raw bars
# until close * volume reaches the dollar threshold.
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["features"]["bar"] = {
    "type": "dollar",
    "dollar_threshold": 1_000_000,
    "history_window": 390,
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7I. Change CatBoost model path
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["paths"]["catboost_model_path"] = "models/catboost_model.cbm"

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7J. Change CatBoost feature columns
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["model"]["feature_cols"] = [
    "bb_score",
    "rsi",
    "session_minutes",
    "minutes_until_close",
    # add the exact feature names your CatBoost model expects
]

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7K. Change double-down rule
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["double_down"]["enabled"] = True
DEFAULT_LIVE_CONFIG["double_down"]["bb_mult"] = 3
DEFAULT_LIVE_CONFIG["double_down"]["discount_pct"] = 0.01

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7L. Disable double-down
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["double_down"]["enabled"] = False

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7M. Change sizing schedule
#
# active_n means:
#   0 open trades -> qty 100
#   1 open trade  -> qty 100
#   2 open trades -> qty 200
#   3 open trades -> qty 300
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["sizing"]["default_qty"] = 100

DEFAULT_LIVE_CONFIG["sizing"]["size_schedule"] = {
    0: 100,
    1: 100,
    2: 200,
    3: 300,
}

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7N. Change entry / risk limits
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["max_open_trades_today"] = 4
DEFAULT_LIVE_CONFIG["strategy"]["max_new_trades_per_day"] = 4
DEFAULT_LIVE_CONFIG["strategy"]["max_open_trades_total"] = 200

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7O. Change timing rules
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["no_trade_first_minutes"] = 60
DEFAULT_LIVE_CONFIG["strategy"]["no_new_entries_last_minutes"] = 60

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7P. Change take-profit
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["min_take_profit"] = 0.01

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7Q. Change reentry cooldown after sells
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["reentry_cooldown_days"] = 7
DEFAULT_LIVE_CONFIG["strategy"]["reentry_discount_pct"] = 0.01

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7R. Change limit order entry offset
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["strategy"]["limit_entry_offset_pct"] = 0.0005

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7S. Change open-trades CSV path
#
# This is where unfinished trades are saved/reloaded.
# On restart, the bot loads this file and continues managing them.
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["paths"]["open_trades_path"] = "AMD_open_trades.csv"

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7T. Turn autosave on/off
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["saving"]["autosave_open_trades"] = True

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 7U. Silence noisy logs
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["debug"]["bars"] = False
DEFAULT_LIVE_CONFIG["debug"]["features"] = True
DEFAULT_LIVE_CONFIG["debug"]["signal"] = True
DEFAULT_LIVE_CONFIG["debug"]["orders"] = True
DEFAULT_LIVE_CONFIG["debug"]["save"] = False
DEFAULT_LIVE_CONFIG["debug"]["model"] = True

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)


# ------------------------------------------------------------
# 8. Manual check: inspect loaded unfinished trades
# ------------------------------------------------------------

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)
execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)

order_router = IBKRLimitOrderRouter(
    ib=ib,
    contract=stock,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    execution_instrument=execution_instrument,
)

algo.open_trades


# ------------------------------------------------------------
# 9. Manual save open trades
#
# Useful in notebook after manually editing algo.open_trades.
# ------------------------------------------------------------

algo.open_trades_store.save(algo.open_trades)


# ------------------------------------------------------------
# 10. Manual reload open trades from CSV
# ------------------------------------------------------------

algo.open_trades = algo.open_trades_store.load()

algo.open_trades


# ------------------------------------------------------------
# 11. Remove callback from real_time_bars
#
# Useful when rerunning notebook cells so you do not attach
# the same algo.on_bar multiple times.
# ------------------------------------------------------------

real_time_bars.updateEvent -= algo.on_bar


# ------------------------------------------------------------
# 12. Reattach callback
# ------------------------------------------------------------

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 13. Full clean IBKR default run
# ------------------------------------------------------------

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)
execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)

order_router = IBKRLimitOrderRouter(
    ib=ib,
    contract=stock,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    execution_instrument=execution_instrument,
)

real_time_bars.updateEvent += algo.on_bar


# ------------------------------------------------------------
# 14. Full clean IBKR + CatBoost run
# ------------------------------------------------------------

DEFAULT_LIVE_CONFIG["model"]["enabled"] = True
DEFAULT_LIVE_CONFIG["model"]["prob_threshold"] = 0.55

config = LiveTradingConfig.from_dict(DEFAULT_LIVE_CONFIG)
execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)

model_cfg = config.model
paths_cfg = config.paths

model_filter = CatBoostLiveModelFilter(
    model_path=paths_cfg["catboost_model_path"],
    feature_cols=model_cfg["feature_cols"],
    prob_threshold=model_cfg.get("prob_threshold", 0.55),
    enabled=model_cfg.get("enabled", True),
)

order_router = IBKRLimitOrderRouter(
    ib=ib,
    contract=stock,
)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    model_filter=model_filter,
    execution_instrument=execution_instrument,
)

real_time_bars.updateEvent += algo.on_bar